### Subqueries in `WHERE`

These are query in a query. Parantheses are used in the `WHERE` clause.

```sql
SELECT name, total_viewers
FROM sports
WHERE total_viewers < (SELECT AVG(total_viewers) FROM sports)
ORDER BY total_viewers DESC

### Subqueries in `JOIN`

We can also do it in the `JOIN` to create a temporary table to be queried.

```sql
SELECT e.name, e.salary
FROM employees e

-- joining a avg_salary table
JOIN (
  SELECT AVG(salary) AS avg_salary
  FROM employees
  WHERE department = 'Finance'
) AS avg
ON e.salary > avg.avg_salary;
```

```python
# output
name	salary	avg_salary
Alice	50000	55000
Bob	60000	55000

```sql
-- Another example

SELECT name, salary
FROM employees
JOIN (
    SELECT AVG(salary) AS avg_salary
    FROM employees
    WHERE department = 'marketing'
) AS marketing_average

-- ON is being used as a filter condition
ON salary<marketing_average.avg_salary

WHERE department = 'marketing'
ORDER BY salary ASC

### `EXISTS` and `NOT EXISTS` Operator

It is used to check if a subquery returns any rows. If the subquery returns any rows, the `EXISTS` operator return `TRUE`, else it returns `FALSE`

```sql
SELECT u.name
FROM users u
WHERE NOT EXISTS(
    -- checking for existence of 1 is enough
    SELECT 1
    FROM posts p
    WHERE u.id = p.user_id
)
ORDER BY u.name ASC


### `ANY` and `ALL` Operator

`ANY` is useful for dealing with multiple values.

```sql
-- this returns customers who have made an order with a price higher than 150
SELECT name
FROM customers
WHERE id = ANY (
    SELECT DISTINCT customer_id
    FROM orders
    WHERE price > 150
);

-- another example
SELECT c.name
FROM customers c
WHERE c.id = ANY(
    SELECT o.id
    FROM orders o
    WHERE o.price<100
)
```

`ALL` operator works similarly

### Common Table Expressions (CTE)

With PostgresSQL, you can temporarily save the results of a query to a CTE (Common Table Expressions). THis is useful when you want to reuse the results of a query in multiple places in your query.

These are defined using the `WITH` caluse

```sql 

-- declared here
WITH high_value_orders AS (
    SELECT customer_id, SUM(price) as total_spend
    FROM orders
    GROUP BY customer_id
    HAVING SUM(price) > 1000
)

SELECT c.name, hvo.total_spend
FROM customers c

-- used here
JOIN high_value_orders hvo ON c.id = hvo.customer_id
ORDER BY hvo.total_spend DESC;
```

A CTE isnt persisted in the database and is only used for the duration of the query. It shouldnt have a semicolon at the end!

```sql
-- temporarilty creating a table containig each customers' id and their max purchase
WITH distinct_max AS 
    (SELECT customer_id, max(price) as max_price 
    FROM orders
    GROUP BY customer_id)

-- joining the tables together and checking which max purchases is lower than 100
SELECT c.name
FROM customers c
JOIN distinct_max dm ON c.id = dm.customer_id
WHERE dm.max_price < 100
ORDER BY c.name ASC

### CASE Expression

This is similar to an `if-else` statement in other programming languages. This is used in the `SELECT` clause.

The syntax is as follow
```sql
CASE
    -- different switch case
    WHEN salary > 100000 THEN 'High Salary'
    ELSE 'Low Salary'

-- ending as a new col
END AS salary_level
```


```sql
-- another example

SELECT name, 
CASE 
    WHEN department = 'Engineering' THEN 'yes'
    else 'no'
END AS is_engineering
FROM employees
ORDER BY name
